<div align="center">

# Patra Toolkit: Model Cards & Datasheets

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Data-to-Insight-Center/patra-toolkit/blob/main/examples/notebooks/ModelCardAndDatasheetDemo.ipynb)

</div>

[Patra](https://github.com/Data-to-Insight-Center/patra-knowledge-base) documents AI/ML models and the datasets that trained them as structured, machine-actionable metadata -- **Model Cards** and **Datasheets** -- instead of a README that goes stale. This notebook builds a Model Card and Datasheet, submits them to a Patra server, looks up existing records, and (optionally) streams a live inference run through CKN.

## Install

In [ ]:
!pip install -q patra-toolkit

## Connect to a Patra server

Reads (list/get) work anonymously. Writes (`submit`) need a Tapis token -- get one with `mc.authenticate(username=..., password=...)`.

In [ ]:
patra_server_url = "https://patrabackenddemo.pods.icicleai.tapis.io/"
tapis_token = None

## Build a Model Card

Only `name` is required -- everything else, including the attached `AIModel`, is optional.

In [ ]:
from patra_toolkit import ModelCard, AIModel

mc = ModelCard(
    name="UCI_Adult_Model",
    version="1.0",
    short_description="UCI Adult Data analysis using Tensorflow for demonstration of Patra Model Cards.",
    full_description="We have trained a ML model using the tensorflow framework to predict income for the UCI Adult Dataset. We leverage this data to run the Patra model cards to capture metadata about the model as well as fairness and explainability metrics.",
    keywords="uci adult, tensorflow, explainability, fairness, patra",
    author="0009-0009-9817-7042",
    input_type="Tabular",
    category="classification",
    foundational_model="None",
    citation="Becker, B. & Kohavi, R. (1996). Adult [Dataset]. UCI.",
)
mc.input_data = "https://archive.ics.uci.edu/dataset/2/adult"
mc.output_data = "https://huggingface.co/patra-iu/neelk-uci_adult_model-1.0"

mc.ai_model = AIModel(
    name="Random Forest",
    version="0.1",
    description="Census classification problem using Random Forest",
    owner="neelk",
    location="https://github.iu.edu/swithana/mcwork/randomforest/adult_model.pkl",
    license="BSD-3 Clause",
    framework="sklearn",
    model_type="random_forest",
    test_accuracy=0.85,
)

mc.validate()

## Build a Datasheet

Documents the dataset the model was trained on, the same way.

In [ ]:
from patra_toolkit import Datasheet

ds = Datasheet(publication_year=2025, version="1.0")
ds.add_title("UCI Adult Dataset")
ds.add_creator("Becker, B.")
ds.add_creator("Kohavi, R.")
ds.add_description("Predict whether income exceeds $50K/yr based on census data.", "Abstract")

ds.validate()

## Submit

Raises `PatraModelExistsError`/`PatraDatasheetExistsError` if an equivalent record already exists -- bump `version` to submit a new one.

In [ ]:
if tapis_token:
    mc.submit(patra_server_url=patra_server_url, token=tapis_token)
    ds.submit(patra_server_url=patra_server_url, token=tapis_token)
    print(f"Model Card: {mc.uuid}\nDatasheet: {ds.uuid}")
else:
    print("No token set -- call mc.authenticate(username=..., password=...) to get one and actually submit.")

## Discover existing records

`list_*` also takes `q=` (substring search) and `skip=`/`limit=` for paging.

In [ ]:
import pandas as pd

model_cards = ModelCard.list_model_cards(server_url=patra_server_url, token=tapis_token, limit=5)
pd.DataFrame(model_cards)

In [ ]:
ModelCard.get_model_card(server_url=patra_server_url, uuid=model_cards[0]["uuid"], token=tapis_token)

In [ ]:
datasheets = Datasheet.list_datasheets(server_url=patra_server_url, token=tapis_token, limit=5)
pd.DataFrame(datasheets)

In [ ]:
Datasheet.get_datasheet(server_url=patra_server_url, uuid=datasheets[0]["uuid"], token=tapis_token)

## Run a live inference experiment (optional)

`run_experiment()` fetches a Model Card + Datasheet, downloads the model and sample images they reference, runs inference, and streams a metric event per image to CKN (Kafka) so results show up in Patra's web app live. This uses different, pre-existing demo records with real downloadable weights/images -- not the ones built above.

In [ ]:
!pip install -q "patra-toolkit[experiments]"

In [ ]:
import torchvision
from patra_toolkit import run_experiment

model_card_uuid = "56e0fc98-f6dd-4994-a6e3-dc6ca0f4f5e6"  # pretrained ResNet50
datasheet_uuid = "d169e2ed-2435-49f0-93ef-207abcc44ede"  # Lorem Picsum sample images
categories = torchvision.models.ResNet50_Weights.IMAGENET1K_V2.meta["categories"]

Registering is idempotent -- safe to re-run.

In [ ]:
import requests

user_id = "demo_user"  # replace with your own

requests.post(f"{patra_server_url.rstrip('/')}/users", json={"username": user_id}).raise_for_status()

In [ ]:
result = run_experiment(
    model_card_uuid=model_card_uuid,
    datasheet_uuid=datasheet_uuid,
    patra_server_url=patra_server_url,
    ckn_broker_url="cknbroker.pods.icicleai.tapis.io:443",  # your broker's advertised external address
    user_id=user_id,
    token=tapis_token,
    categories=categories,
)
result

View results at `https://patrabackend.pods.icicleai.tapis.io/experiments/digital-ag/users/demo_user/summary` -- the server CKN's sink connector actually writes to, which may differ from `patra_server_url` -- or in the Patra web app at **[patra.pods.icicleai.tapis.io](https://patra.pods.icicleai.tapis.io)**, under **Experiments → Digital Agriculture**. Note: `patrademo`/`patrabackenddemo` is a separate demo instance CKN never writes to -- it will never show this data.

## Next steps

- **Fairness & explainability**: `mc.populate_bias(...)` ([fairlearn](https://fairlearn.org/)) and `mc.populate_xai(...)` ([SHAP](https://shap.readthedocs.io/)).
- **Full field reference**: [schema_description.md](https://github.com/Data-to-Insight-Center/patra-toolkit/blob/main/docs/source/schema_description.md).
- **More examples**: [examples/notebooks/](https://github.com/Data-to-Insight-Center/patra-toolkit/tree/main/examples/notebooks).
- **Browse visually**: the Patra web app (`patra-frontend/` in this workspace).